# 15 — V3-E3: probes temporais MiniLM e TF-IDF

Este notebook apresenta a evidência agregada do V3-E3. Ele é somente leitura: não treina, não recalibra, não procura novos thresholds e não reabre as partições científicas. A auditoria valida o resultado já produzido e seu manifesto antes de exibir qualquer métrica.

## Contrato executado

Foram comparados MiniLM congelado + LinearSVC e TF-IDF word/char + LinearSVC nas razões negativas 1:5, 1:10 e 1:15. Todos os positivos críticos únicos foram mantidos. Os negativos combinaram exemplos difíceis minerados fora da amostra com uma amostra de fundo estratificada por mês e família.

Cada configuração enfrentou três folds rolling-origin. O calibrador e o threshold foram ajustados somente na janela de calibração imediatamente futura; a avaliação seguinte manteve a prevalência natural e não forneceu um threshold oráculo. A precisão crítica mínima congelada foi 0,40.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.v3.e3_reporting import (
    audit_v3_e3_results,
    render_v3_e3_audit,
)

## Resultado auditado

A célula abaixo verifica hashes e tamanhos, fronteiras científicas, catálogo de candidatos, contagens e assinaturas das amostras, suporte dos folds, métricas limitadas, elegibilidade, ranking congelado, seleção e gates da comparação S7. Nenhum texto, embedding, score individual ou estimador ajustado é carregado.

In [2]:
audit = audit_v3_e3_results(PROJECT_ROOT)
print(render_v3_e3_audit(audit, 'pt-BR'))

V3-E3 — RESULTADOS TEMPORAIS DOS PROBES

RESUMO
  status: VALID_COMPLETE_DEVELOPMENT_RESULTS
  selected_candidate: tfidf_word_char_ratio_1_10
  eligible_candidates: 3
  runtime_seconds: 11800.66
  result_sha256: DCBE983B492E09418980185827829531EDA26C755FA184B58440B7F7BE715391
  manifest_sha256: EB33B8DDEBE2F1A7F4FDEFBD9CE493268FA6224F12E64F410BF26A343DA66E53

RANKING CONGELADO
  1. tfidf_word_char_ratio_1_10 | eligible=True | worst_recall@P>=0.40=0.1370 | median_PR-AUC=0.2258 | median_F1=0.2914
  2. tfidf_word_char_ratio_1_15 | eligible=True | worst_recall@P>=0.40=0.1096 | median_PR-AUC=0.2463 | median_F1=0.3322
  3. tfidf_word_char_ratio_1_5 | eligible=True | worst_recall@P>=0.40=0.0320 | median_PR-AUC=0.1284 | median_F1=0.1660
  4. minilm_frozen_ratio_1_15 | eligible=False | worst_recall@P>=0.40=0.0000 | median_PR-AUC=0.1111 | median_F1=0.2074
  5. minilm_frozen_ratio_1_10 | eligible=False | worst_recall@P>=0.40=0.0000 | median_PR-AUC=0.0889 | median_F1=0.1549
  6. minilm_frozen_rati

## Como interpretar

A primeira regra de seleção é o pior recall crítico entre os folds que preservam precisão de pelo menos 0,40. Essa regra favorece estabilidade temporal, não o maior F1 numa única janela. PR-AUC mediana, F1 mediano, amplitude de F1, tempo e identificador entram apenas como desempates sucessivos.

A comparação com S7 usa somente F3, como congelado. O detector selecionado atua como uma camada de override crítico sobre as previsões S7; os resultados do sistema híbrido e do S7 isolado são medidos nas mesmas 127.706 observações futuras.

Mesmo que os gates de empacotamento de desenvolvimento sejam aprovados, este notebook não autoriza teste confirmatório, stress, monitoramento ou implantação.